# Evidence: the `Measurement` record and the closed forms behind the prior route

`axiom.calibrate` folds external evidence — typically a randomized experiment — into a
response-surface model. Everything starts from one typed record. A **`Measurement`** is a point
estimate with a standard error *and* the `Estimand` it estimates. The estimand carries every
scope facet (intervention doses, population, window, level, ...) and the unit of the estimate,
so two measurements are "the same quantity" exactly when their estimands have the same content
hash, and a gap between a measurement and what the surface realizes is a typed `TransferPlan`,
never a silent reinterpretation.

Nothing in this notebook touches a surface. It shows the record and the four closed forms the
prior route composes:

| function | formula |
|---|---|
| `combine_inverse_variance(t, se)` | `w_i = 1/se_i²`, `mean = Σ w_i t_i / Σ w_i`, `se = sqrt(1/Σ w_i)` |
| `design_factor(beta, contribution)` | `mean(contribution / beta)` over paired draws |
| `mean_sd_to_gamma(mean, sd)` | `shape = (mean/sd)²`, `rate = mean/sd²` |
| `lognormal_from_moments(mean, sd)` | `sigma = sqrt(log(1 + (sd/mean)²))`, `mu = log(mean) − sigma²/2` |

In [ ]:
import numpy as np

from axiom.calibrate import (
    Measurement, PriorFamily, amplitude_prior, combine_inverse_variance, design_factor,
    lognormal_from_moments, mean_sd_to_gamma,
)
from axiom.core import D, Assumption, Intervention, Outcome, Population, Spec, TimeWindow, Treatment
from axiom.estimands import Estimand, Level, Quantity

from axiom.display import enable

enable();  # every axiom result renders itself from here on

## The record

The estimand below is a *contrast*: the cumulative outcome over eight periods at dose 100 against
dose 0, at cluster level, in the population `north`. The `Measurement` says a
difference-in-differences design over 40 units estimated it at `1.2 ± 0.3` (a Wald summary at
95% mass). `source` is the provenance string the ledger will quote; `assumptions` are the
design's own licensing assumptions.

In [ ]:
fertilizer = Treatment(name="fertilizer", dimension=D.currency, unit="USD")
yield_total = Outcome(name="yield_total", dimension=D.outcome, unit="kg")

lift = Estimand(
    name="lift_at_100",
    quantity=Quantity(kind="contrast"),
    treatment=fertilizer,
    intervention=Intervention(doses={"fertilizer": 100.0}),
    reference=Intervention(doses={"fertilizer": 0.0}),
    outcome=yield_total,
    population=Population(name="north"),
    window=TimeWindow(start=0, stop=8, basis="cumulative"),
    level=Level(unit="cluster"),
    dimension=D.outcome,
)
parallel_trends = Assumption(
    name="parallel_trends", facet="population",
    statement="treated and control clusters would have moved in parallel without the dose",
    challenged_by="a pre-period placebo test",
)
m1 = Measurement(
    estimand=lift, estimate=1.2, se=0.3, definition="wald", mass=0.95,
    method="difference_in_differences", n_units=40, n_periods=8,
    source="study-1", assumptions=(parallel_trends,),
)
print(m1.interval)
print("precision 1/se² =", round(m1.precision, 3), "| target hash:", m1.target[:12], "...")
print("round-trips:", Spec.from_json(m1.to_json()) == m1)

`interval` is a `core.Interval` (rule 4: every interval carries its definition and mass). The
record holds a `(mean, se)` pair, so the interval is the normal approximation for every
`definition`; the label is kept so the ledger shows how the study reported it rather than
relabelling an `eti` as Wald.

In [ ]:
m2 = Measurement(estimand=lift, estimate=0.9, se=0.5, definition="eti", mass=0.9, source="study-2")
print(m2.interval)
print("same target estimand:", m1.target == m2.target)

## Fixed-effect pooling

Independent estimates of one quantity pool by inverse variance. The pooled standard error is
always at most the smallest input standard error, and a single estimate pools to itself.

In [ ]:
mean, se = combine_inverse_variance([m1.estimate, m2.estimate], [m1.se, m2.se])
print(f"pooled: {mean:.4f} ± {se:.4f}")
w = np.array([m1.precision, m2.precision])
print("by hand:", round(float(np.sum(w * [1.2, 0.9]) / w.sum()), 4), round(float(np.sqrt(1 / w.sum())), 4))
print("one estimate pools to itself:", combine_inverse_variance([1.2], [0.3]))

## The design factor

The surface's amplitude `beta` is not the measured quantity. The realized estimand is
`contribution = g(beta, k, s, doses, ...)`, and at fixed shape and scale it is proportional to
`beta`. The **design factor** is that proportionality measured on the posterior: the *mean of
per-draw ratios* `mean(contribution / beta)`. This is the form that reproduces the parent's golden
value at `1e-12` (ratio of means, median of ratios and regression through the origin were tried and
rejected — see the `calibrate.prior` docstring). Dividing a `(mean, se)` on the target by the
factor gives the implied `(mean, sd)` of the amplitude.

In [ ]:
rng = np.random.default_rng(0)
beta_draws = rng.lognormal(0.0, 0.2, size=2000)
contribution_draws = 1000.0 * beta_draws * rng.normal(1.0, 0.02, size=2000)
f = design_factor(beta_draws, contribution_draws)
print("design factor:", round(f, 3))
print("ratio of means (not used):", round(float(contribution_draws.mean() / beta_draws.mean()), 3))
amp_mean, amp_sd = mean / f, se / f
print(f"implied amplitude: {amp_mean:.5f} ± {amp_sd:.5f}")

## Moment matching to a positive-support family

An amplitude is positive, so the evidence is matched to a **lognormal** or a **gamma** with the
implied first two moments. `amplitude_prior` returns a `core.Prior`; `PriorFamily` is the literal
type of its `family` argument. Both constructions below recover the target mean and sd exactly.

In [ ]:
mu, sigma = lognormal_from_moments(amp_mean, amp_sd)
print(f"lognormal: mu={mu:.5f} sigma={sigma:.5f}")
print("  mean back:", round(float(np.exp(mu + sigma**2 / 2)), 6), "sd back:",
      round(float(np.sqrt((np.exp(sigma**2) - 1) * np.exp(2 * mu + sigma**2))), 6))

shape, rate = mean_sd_to_gamma(amp_mean, amp_sd)
print(f"gamma: shape={shape:.4f} rate={rate:.4f}")
print("  mean back:", round(shape / rate, 6), "sd back:", round(float(np.sqrt(shape) / rate), 6))

for fam in ("lognormal", "gamma"):
    family: PriorFamily = fam
    print(amplitude_prior(amp_mean, amp_sd, family))

Every one of these functions raises `ValueError` on an empty, mismatched or non-positive input
rather than returning a number that looks plausible — the prior route never sees a silent NaN.

In [ ]:
for bad in (lambda: combine_inverse_variance([1.0, 2.0], [0.1, -0.1]),
            lambda: design_factor([1.0, 0.0], [1.0, 1.0]),
            lambda: mean_sd_to_gamma(-1.0, 0.5)):
    try:
        bad()
    except ValueError as e:
        print("ValueError:", e)